# Fock v2.1 Creation Gate Routing Fix

## Motivation

The register diversity/entropy eval on the converged Fock v2 Multi-Xi checkpoint
(12.00 PPL) diagnosed **temperature collapse**: after layer 0, every register's
creation attention collapsed to essentially a single token (α_max > 0.99,
entropy < 0.005).  The overall mean entropy was 0.04 (target: 0.30 for genuine
routing) and mean diversity was 0.37 (target: 0.79).

Verdict: **MIXED — temperature collapse, not mean-pooling.**

This notebook implements **Step 1 of the resolution hierarchy** (§5.2 of
Fock-PARFLM_Next_Steps.md): fix the creation gate via three interventions:

| Fix | Description | Params added |
|-----|-------------|------|
| **B1** | Per-register learnable temperature (each register has its own τ_k) | M |
| **B2** | Per-register key subspaces (W_K ∈ ℝ^{M×d×d_k} instead of shared) | M·d·d_k |
| **B3** | Orthogonal register embedding initialisation | 0 |

## Baselines

| Model | PPL | Memory |
|-------|-----|--------|
| Multi-ξ K=4 conservative base | 12.47 | O(1) |
| Fock v2 registers (K=4, M=16) | 12.00 | O(1) |
| Fock Attention 4-head (direct exchange) | 10.93 | O(T²) |
| Matched attention baseline (MatchedGPT) | 7.81 | O(T²) |

## Arms (5-arm ablation)

| # | Arm | B1 (τ_k) | B2 (K_k) | B3 (ortho) | Steps | Description |
|---|-----|:---:|:---:|:---:|---|---|
| 1 | `v2_baseline_rerun` | ✗ | ✗ | ✗ | 16k | Exact v2 rerun (control) |
| 2 | `v21_tau_only` | ✓ | ✗ | ✗ | 16k | Per-register temperature only |
| 3 | `v21_tau_perK` | ✓ | ✓ | ✗ | 16k | Per-register τ + keys |
| 4 | `v21_tau_perK_ortho` | ✓ | ✓ | ✓ | 16k | All three fixes |
| 5 | `v21_ortho_only` | ✗ | ✗ | ✓ | 16k | Orthogonal init only |

All arms use τ_create_init = √d_k = 8.0 (standard 1/√d_k scaling) instead
of the original 0.1 (which caused the temperature collapse).

## Hardware

- **T4 HighMem / A100 40GB / H100 80GB**: Per-register keys add ~M·d·d_k ≈ 16·256·64 = 262K params.
  Total overhead is modest. Same memory profile as v2.
- No grad-accum needed
- TF32 disabled for autograd.grad stability

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_v21_routing_fix')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_fock_v21_routing_fix'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
PARF_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

All arms use the same base config as the Fock v2 16k run that achieved 12.00 PPL:
- `--use-layer-checkpoint` (Level-2)
- `--use-gathered-v-phi` (Stage-1.5b)
- `--v-phi-phi-hidden 128 --v-phi-theta-hidden 128`
- `--ln-before-distance --per-layer-v-phi-scale` (P8)
- `--xi-alpha-init-mode log_spaced` K=4
- `--gumbel-tau-min 0.3`
- `--fock-grad-clip 0.5`
- `--fock-version v2 --n-registers 16 --stack-discipline --reverse-channel`

The v2.1 fixes are enabled via:
- `--per-register-tau` (B1)
- `--per-register-keys` (B2)
- `--ortho-register-init` (B3)
- `--tau-create-init 8.0` (√d_k = √64 = 8 for standard-scale attention)

In [ ]:
MODE           = 'scaleup'
FIXED_GAMMA    = 0.30
SEED           = 0
MAX_TRAIN_TOK  = 5_000_000
V_PHI_H        = 128
GUMBEL_TAU_MIN = 0.3
FOCK_GRAD_CLIP = 0.5
MAX_STEPS      = 16000

# ── Causal leak fix ──────────────────────────────────────────────────
PREFIX_CAUSAL  = True    # prefix-causal register lifecycle (leak fix)
CAUSAL_PROBE_INTERVAL       = 4000   # architectural probe every N steps (0=off)
TRAINED_LEAK_PROBE_INTERVAL = 8000   # trained-scale probe + honest PPL every N steps (0=off)
TRAINED_LEAK_PROBE_K        = 256    # honest-PPL target tokens (256=fast, 1024=thorough)
TRAINED_LEAK_PROBE_PAIRS    = 2      # future-perturbation window pairs

# ── Eval memory guard ────────────────────────────────────────────────
# Eval builds a full autograd graph (the physics force computation needs
# torch.autograd.grad even in eval mode) but never calls .backward(), so it
# never gets the memory relief that --use-layer-checkpoint provides during
# training. Chunk eval batches to this size to avoid CUDA OOM on T4/L4.
EVAL_MICRO_BATCH = 4

# ── Training memory guard ────────────────────────────────────────────
# The prefix-causal fix expands register tensors from (B,M,d) to (B,T,M,d),
# increasing per-layer memory by ~512x for B=16,T=512. With grad_accum=4
# the micro-batch drops from 16 to 4, keeping effective batch constant while
# fitting comfortably on T4 (15GB). Set to 1 for larger GPUs (L4/A100).
GRAD_ACCUM = 4

# τ_create_init for v2.1 arms: √d_k for standard 1/√d_k scaling
TAU_V21 = 8.0   # sqrt(64)
# τ_create_init for v2 baseline rerun: original value
TAU_V2  = 0.1

ARM_DEFS = {
    'v2_baseline_rerun': {
        'per_register_tau': False,
        'per_register_keys': False,
        'ortho_register_init': False,
        'tau_create_init': TAU_V2,
        'desc': 'Exact v2 rerun at 16k steps (control) — same as original 12.00 PPL run',
    },
    'v21_tau_only': {
        'per_register_tau': True,
        'per_register_keys': False,
        'ortho_register_init': False,
        'tau_create_init': TAU_V21,
        'desc': 'B1 only: per-register learnable temperature (highest leverage)',
    },
    'v21_tau_perK': {
        'per_register_tau': True,
        'per_register_keys': True,
        'ortho_register_init': False,
        'tau_create_init': TAU_V21,
        'desc': 'B1 + B2: per-register τ + per-register key subspaces',
    },
    'v21_tau_perK_ortho': {
        'per_register_tau': True,
        'per_register_keys': True,
        'ortho_register_init': True,
        'tau_create_init': TAU_V21,
        'desc': 'B1 + B2 + B3: all three fixes (per-reg τ, per-reg K, ortho init)',
    },
    'v21_ortho_only': {
        'per_register_tau': False,
        'per_register_keys': False,
        'ortho_register_init': True,
        'tau_create_init': TAU_V2,
        'desc': 'B3 only: orthogonal register init (zero parameter cost baseline)',
    },
}

ALL_ARMS = list(ARM_DEFS.keys())

print(f'Arms defined : {len(ALL_ARMS)}')
for name in ALL_ARMS:
    d = ARM_DEFS[name]
    flags = []
    if d['per_register_tau']:   flags.append('B1:per_reg_\u03c4')
    if d['per_register_keys']:  flags.append('B2:per_reg_K')
    if d['ortho_register_init']: flags.append('B3:ortho')
    flag_str = ', '.join(flags) if flags else '(none — v2 baseline)'
    print(f'  {name:25s}  \u03c4_init={d["tau_create_init"]:5.1f}  fixes=[{flag_str}]')
    print(f'  {"":25s}  {d["desc"]}')

In [ ]:
# ── SELECT ARMS TO RUN ────────────────────────────────────────────────────
# Edit ARMS_TO_RUN to target any subset.
#
# Recommended order (most informative first):
#   1. v21_tau_only        — B1 alone; if this works, temperature was the bottleneck
#   2. v2_baseline_rerun   — control to confirm reproducibility
#   3. v21_tau_perK_ortho  — all fixes; upper bound for Step 1
#   4. v21_tau_perK        — isolate B3 contribution
#   5. v21_ortho_only      — isolate B3 without tau fix
# ──────────────────────────────────────────────────────────────────────────

if 'results' not in dir():
    results = {}

#ARMS_TO_RUN = ALL_ARMS
ARMS_TO_RUN = ['v21_tau_perK_ortho']

invalid = [n for n in ARMS_TO_RUN if n not in ARM_DEFS]
if invalid:
    raise ValueError(f'Unknown arm names: {invalid}\nValid: {ALL_ARMS}')

print(f'Schedule     : {MODE}  max_steps={MAX_STEPS}')
print(f'Stability    : gumbel_tau_min={GUMBEL_TAU_MIN}  fock_grad_clip={FOCK_GRAD_CLIP}')
print()
print(f'Selected {len(ARMS_TO_RUN)} / {len(ALL_ARMS)} arms:')
for name in ARMS_TO_RUN:
    d = ARM_DEFS[name]
    flags = []
    if d['per_register_tau']:   flags.append('\u03c4_k')
    if d['per_register_keys']:  flags.append('K_k')
    if d['ortho_register_init']: flags.append('ortho')
    print(f'  {name:25s}  fixes=[{", ".join(flags) or "none"}]')

## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists()
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Train Fock v2.1 arms

Each arm trains `FockMultiXiPARFLM` with the v2.1 creation gate fixes.
The training script is `train_fock_multixi_scaleup.py` with the new CLI flags.

Completed arms are skipped on re-run.

In [ ]:
import re
from IPython.display import clear_output

_TRAIN_RE = re.compile(
    r'step\s+(\d+)/(\d+)\s+train\s+([\d.]+)\s+lr\s+([\d.eE+-]+)\s+'
    r'grad\s+([\d.]+)\s+'
    r'(?:\(fock=([\d.]+)\s+vphi=([\d.]+)\)\s+)?'
    r'gamma=([\d.]+)\s+tau=([\d.]+)'
    r'(?:\s+fock_tau=([\d.]+(?:\[[\d.,]+\])?))?'
    r'(?:\s+rev_s=([\d.eE+-]+))?'
    r'\s+[^\[]*\[([^\]]+)\]\s+elapsed\s+([\d.]+)s'
)
_EVAL_RE = re.compile(
    r'>>>\s+eval\s+@\s+(\d+):\s+val\s+([\d.]+)\s+ppl\s+([\d.]+)'
)

def _parse_train(line):
    m = _TRAIN_RE.search(line)
    if not m:
        return None
    d = {
        'step': int(m.group(1)), 'total': int(m.group(2)),
        'train_loss': float(m.group(3)), 'lr': float(m.group(4)),
        'grad': float(m.group(5)),
        'gamma': float(m.group(8)), 'tau': float(m.group(9)),
        'alphas': m.group(12),
        'elapsed_s': float(m.group(13)),
    }
    if m.group(6) is not None:
        d['grad_fock'] = float(m.group(6))
    if m.group(7) is not None:
        d['grad_vphi'] = float(m.group(7))
    if m.group(10) is not None:
        d['fock_tau'] = m.group(10)
    if m.group(11) is not None:
        d['rev_s'] = float(m.group(11))
    return d

def _parse_eval(line):
    m = _EVAL_RE.search(line)
    if not m:
        return None
    return {'step': int(m.group(1)), 'val_loss': float(m.group(2)),
            'val_ppl': float(m.group(3))}

def _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=False):
    clear_output(wait=True)
    import sys as _sys; _sys.stdout.flush()
    status = '\u2713 DONE' if done else '\u2699 RUNNING'
    flags = []
    if arm_def['per_register_tau']:   flags.append('B1:\u03c4_k')
    if arm_def['per_register_keys']:  flags.append('B2:K_k')
    if arm_def['ortho_register_init']: flags.append('B3:ortho')
    flag_str = ', '.join(flags) if flags else 'none (v2 baseline)'
    W = 65
    print('\u2550' * W)
    print(f'  {status}  {arm_name}')
    print(f'  Fixes: [{flag_str}]  \u03c4_init={arm_def["tau_create_init"]}')
    print(f'  {arm_def["desc"]}')
    print('\u2550' * W)

    if tr:
        step, total = tr['step'], tr['total']
        elapsed = time.time() - t_start
        train_elapsed = (time.time() - t_train_start) if t_train_start else elapsed
        pct = 100 * step / total
        eta_s = train_elapsed * (total - step) / step if step > 0 else 0
        eta_str = 'Done!' if done else f'ETA {eta_s / 60:.1f} min'
        bar_w = 52
        filled = int(bar_w * pct / 100)
        bar = '\u2588' * filled + '\u2591' * (bar_w - filled)
        print(f'  [{bar}]')
        print(f'  Step : {step:5d} / {total}  ({pct:.1f}%)'
              f'   Elapsed: {elapsed / 60:.1f} min   {eta_str}')
        print()
        grad_detail = ''
        if 'grad_fock' in tr:
            grad_detail = f'  (fock={tr["grad_fock"]:.1f}  vphi={tr["grad_vphi"]:.1f})'
        print(f'  train loss : {tr["train_loss"]:.4f}'
              f'   lr : {tr["lr"]:.2e}'
              f'   grad norm : {tr["grad"]:.3f}{grad_detail}')
        fock_tau_str = ''
        if 'fock_tau' in tr:
            fock_tau_str = f'   fock \u03c4_create: {tr["fock_tau"]}'
        rev_str = ''
        if 'rev_s' in tr:
            rev_str = f'   rev scale: {tr["rev_s"]:.3f}'
        print(f'  \u03b3          : {tr["gamma"]:.4f}'
              f'   \u03c4 (Gumbel): {tr["tau"]:.4f}'
              f'{fock_tau_str}{rev_str}')
        print(f'  \u03b1 channels : [{tr["alphas"]}]')

    if ev:
        print()
        print(f'  Last val PPL : {ev["val_ppl"]:.3f}'
              f'   val loss : {ev["val_loss"]:.4f}'
              f'   (@ step {ev["step"]})')

    noise = [l for l in recent_lines
             if l.strip()
             and '[fock-multixi-parf] step' not in l
             and not l.startswith('{')]
    if noise:
        print()
        print('  Recent output:')
        for l in noise[-6:]:
            print(f'    {l}')

def _run_arm_streaming(arm_name, arm_def, cmd):
    all_lines, recent_lines = [], []
    tr, ev = None, None
    t_start = time.time()
    t_train_start = None

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        cwd=str(SCRIPTS_DIR),
        env={**__import__('os').environ, 'PYTHONUNBUFFERED': '1'},
    )
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            all_lines.append(line)
            recent_lines = all_lines[-40:]

            new_tr = _parse_train(line)
            new_ev = _parse_eval(line)
            if new_tr:
                tr = new_tr
                if t_train_start is None:
                    t_train_start = time.time()
            if new_ev:
                ev = new_ev
            if new_tr or new_ev:
                _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines)
            elif tr is None:
                if line.strip() and not line.startswith('{'):
                    print(line, flush=True)
    except Exception as exc:
        print(f'\n  STREAM ERROR: {exc}')
    finally:
        proc.wait()

    _show_progress(arm_name, arm_def, tr, ev, t_start, t_train_start, recent_lines, done=True)
    return proc.returncode, all_lines


# ── Training loop ─────────────────────────────────────────────────────────
TRAINER = str(SCRIPTS_DIR / 'train_fock_multixi_scaleup.py')

for arm_name in ARMS_TO_RUN:
    arm_def = ARM_DEFS[arm_name]
    arm_results_dir = DRIVE_RESULTS / arm_name
    arm_results_dir.mkdir(parents=True, exist_ok=True)

    summary_glob = list(arm_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n\u23ed  {arm_name}: SKIP (already complete)')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final' in line:
                    print(f'   {line.strip()}')
        results[arm_name] = {'status': 'skipped'}
        continue

    cmd = [
        sys.executable, '-u', TRAINER,
        '--mode', MODE,
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
        '--v-phi-kind', 'structural_competitive',
        '--top-k', '8',
        '--v-phi-phi-hidden', str(V_PHI_H),
        '--v-phi-theta-hidden', str(V_PHI_H),
        '--use-layer-checkpoint',
        '--use-gathered-v-phi',
        '--ln-before-distance',
        '--per-layer-v-phi-scale',
        '--gumbel-tau-min', str(GUMBEL_TAU_MIN),
        '--fock-grad-clip', str(FOCK_GRAD_CLIP),
        '--xi-channels', '4',
        '--xi-alpha-init-mode', 'log_spaced',
        '--fock-version', 'v2',
        '--n-registers', '16',
        '--stack-discipline',
        '--reverse-channel',
        '--max-steps', str(MAX_STEPS),
        '--tau-create-init', str(arm_def['tau_create_init']),
        '--prefix-causal-registers' if PREFIX_CAUSAL else '--no-prefix-causal-registers',
        '--causal-probe-interval', str(CAUSAL_PROBE_INTERVAL),
        '--trained-leak-probe-interval', str(TRAINED_LEAK_PROBE_INTERVAL),
        '--trained-leak-probe-k', str(TRAINED_LEAK_PROBE_K),
        '--trained-leak-probe-pairs', str(TRAINED_LEAK_PROBE_PAIRS),
        '--eval-micro-batch', str(EVAL_MICRO_BATCH),
        '--grad-accum', str(GRAD_ACCUM),
        '--checkpoint-interval', '1000',
        '--resume',
    ]
    if arm_def['per_register_tau']:
        cmd.append('--per-register-tau')
    if arm_def['per_register_keys']:
        cmd.append('--per-register-keys')
    if arm_def['ortho_register_init']:
        cmd.append('--ortho-register-init')

    rc, all_lines = _run_arm_streaming(arm_name, arm_def, cmd)

    if rc != 0:
        print(f'\n  \u2717 FAILED: trainer exited with code {rc}')
        print('  Last 30 lines of output:')
        for l in all_lines[-30:]:
            print(f'    {l}')
        results[arm_name] = {'status': 'failed', 'returncode': rc}
        continue

    summary_files = list(arm_results_dir.glob('*_summary.md'))
    if summary_files:
        print('\n  \u2500\u2500 Training summary \u2500\u2500')
        with open(summary_files[0]) as f:
            print(f.read())

    results[arm_name] = {'status': 'completed'}

print(f'\n{"\u2550" * 65}')
print(f'All selected arms finished  ({len(ARMS_TO_RUN)} selected)')
for name, r in results.items():
    sym = {'completed': '\u2713', 'skipped': '\u23ed', 'failed': '\u2717'}.get(r['status'], '?')
    print(f'  {sym}  {name:30s}  {r["status"]}')

## 6. Run register diagnostics on completed arms

After training, run the register diversity/entropy eval on each completed
arm's checkpoint to see whether the fixes improved routing.

In [ ]:
DIAG_SCRIPT = str(SCRIPTS_DIR / 'eval_fock_register_diagnostics.py')

for arm_name in ARMS_TO_RUN:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        print(f'  {arm_name}: no checkpoint found, skipping diagnostics')
        continue

    diag_files = list(arm_dir.glob('*_register_diagnostics.json'))
    if diag_files:
        print(f'  {arm_name}: diagnostics already computed, skipping')
        continue

    print(f'\n  Running diagnostics for {arm_name}...')
    # Same memory story as EVAL_MICRO_BATCH: the diagnostic builds an
    # autograd graph over (B,T,M,d) register tensors under PREFIX_CAUSAL,
    # so it runs at the eval micro-batch and takes proportionally more
    # batches to sample the same 160 sequences.
    diag_bs = EVAL_MICRO_BATCH if PREFIX_CAUSAL else 16
    cmd = [
        sys.executable, '-u', DIAG_SCRIPT,
        '--checkpoint', str(ckpt_files[0]),
        '--diag-batches', str(160 // diag_bs),
        '--batch-size', str(diag_bs),
        '--block-size', '512',
        '--device', DEVICE,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(SCRIPTS_DIR))
    if proc.returncode != 0:
        print(f'  FAILED (code {proc.returncode}):')
        print(proc.stderr[-500:] if proc.stderr else proc.stdout[-500:])
    else:
        for line in proc.stdout.splitlines():
            if any(k in line for k in ('VERDICT', 'entropy:', 'diversity:', 'Overall', '---', '===', 'Layer')):
                print(f'  {line}')

print('\nDiagnostics complete.')

## 7. Results comparison

Compare v2.1 arms against v2 baseline and Fock Attention ceiling.

In [ ]:
import matplotlib.pyplot as plt

BASELINES = {
    'Multi-\u03be K=4 conservative': 12.47,
    'Fock v2 M=16 (original)': 12.00,
    'Fock Attention 4h': 10.93,
    'Attention baseline': 7.81,
}

arm_ppls = {}
arm_diag = {}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        continue
    ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
    arm_ppls[arm_name] = ckpt.get('final_val_ppl')

    diag_files = list(arm_dir.glob('*_register_diagnostics.json'))
    if diag_files:
        with open(diag_files[0]) as f:
            arm_diag[arm_name] = json.load(f)

if not arm_ppls:
    print('No results found yet.')
else:
    print(f'{"Arm":25s} {"B1":>3s} {"B2":>3s} {"B3":>3s} '
          f'{"PPL":>8s} {"Entropy":>9s} {"Diversity":>10s} {"\u0394PPL vs v2":>10s}')
    print('\u2500' * 85)
    v2_ppl = BASELINES['Fock v2 M=16 (original)']
    for name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        d = ARM_DEFS[name]
        b1 = '\u2713' if d['per_register_tau'] else ' '
        b2 = '\u2713' if d['per_register_keys'] else ' '
        b3 = '\u2713' if d['ortho_register_init'] else ' '
        ent = arm_diag.get(name, {}).get('overall_mean_entropy', float('nan'))
        div = arm_diag.get(name, {}).get('overall_mean_diversity', float('nan'))
        delta = arm_ppls[name] - v2_ppl
        print(f'{name:25s}  {b1:>3s}  {b2:>3s}  {b3:>3s} '
              f'{arm_ppls[name]:8.2f} {ent:9.4f} {div:10.4f} {delta:+10.2f}')
    print('\u2500' * 85)
    for bname, bppl in BASELINES.items():
        print(f'{bname:25s} {"":>3s} {"":>3s} {"":>3s} {bppl:8.2f}  (baseline)')

    best_arm = min(arm_ppls, key=lambda x: arm_ppls[x])
    best_ppl = arm_ppls[best_arm]
    print(f'\nBest arm: {best_arm} \u2192 {best_ppl:.2f} PPL')
    if best_arm in arm_diag:
        dg = arm_diag[best_arm]
        print(f'  Entropy:  {dg["overall_mean_entropy"]:.4f}  (target: 0.30, was: 0.04)')
        print(f'  Diversity: {dg["overall_mean_diversity"]:.4f}  (target: 0.79, was: 0.37)')

In [ ]:
if arm_ppls:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # PPL bar chart
    ax = axes[0]
    names = list(sorted(arm_ppls, key=lambda x: arm_ppls[x]))
    ppls = [arm_ppls[n] for n in names]
    ax.barh(names, ppls, color='steelblue', edgecolor='white')
    for i, (n, p) in enumerate(zip(names, ppls)):
        ax.text(p + 0.1, i, f'{p:.2f}', va='center', fontsize=9)
    ax.axvline(x=12.00, color='orange', linestyle='--', label='Fock v2 (12.00)')
    ax.axvline(x=10.93, color='green', linestyle='--', label='Fock Attn (10.93)')
    ax.set_xlabel('Val PPL (lower is better)')
    ax.set_title('Fock v2.1 Routing Fix — PPL')
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    ax.grid(True, axis='x', alpha=0.3)

    # Entropy vs Diversity scatter
    ax = axes[1]
    for name in names:
        if name not in arm_diag:
            continue
        dg = arm_diag[name]
        ax.scatter(dg['overall_mean_diversity'], dg['overall_mean_entropy'],
                   s=100, zorder=5, label=f'{name} ({arm_ppls[name]:.1f})')
    ax.scatter(0.785, 0.304, s=150, marker='*', color='green', zorder=10, label='Q6 target')
    ax.scatter(0.145, 1.0, s=150, marker='X', color='red', zorder=10, label='Q0 mean-pool')
    ax.scatter(0.371, 0.040, s=150, marker='s', color='orange', zorder=10, label='v2 original')
    ax.set_xlabel('Register diversity')
    ax.set_ylabel('Normalised attention entropy')
    ax.set_title('Routing Quality Map')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'v21_routing_fix_results.png', dpi=150)
    plt.show()

## 8. Save consolidated report

In [ ]:
report = {
    'experiment': 'fock_v21_routing_fix',
    'motivation': 'Fix temperature collapse in Fock v2 creation gate (entropy 0.04, target 0.30)',
    'config': {
        'mode': MODE,
        'max_steps': MAX_STEPS,
        'fixed_gamma': FIXED_GAMMA,
        'seed': SEED,
    },
    'arms': {},
    'baselines': BASELINES,
}

for name in arm_ppls:
    d = ARM_DEFS[name]
    entry = {
        'per_register_tau': d['per_register_tau'],
        'per_register_keys': d['per_register_keys'],
        'ortho_register_init': d['ortho_register_init'],
        'tau_create_init': d['tau_create_init'],
        'final_ppl': arm_ppls[name],
    }
    if name in arm_diag:
        entry['entropy'] = arm_diag[name]['overall_mean_entropy']
        entry['diversity'] = arm_diag[name]['overall_mean_diversity']
    report['arms'][name] = entry

report_path = DRIVE_RESULTS / 'v21_routing_fix_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
if IN_COLAB:
    print('Results are persisted on Google Drive at:')
    print(f'  {DRIVE_ROOT}')